# LAB: Chaos Agent: Tools, Lore, Questionable Judgment

## Objective

Build a **creative complaint handler** for *NormalObjects* with a modern LangChain agent, then inspect the tool evidence behind its answer.

```text
complaint → agent (LLM) → chooses tools in any order → tool outputs → final answer
```

## Scope

| Section | Work |
|---|---|
| 1 | Setup with `init_chat_model` and `.env` |
| 2 | Four tools from the lab brief |
| 3 | Agent with a grounding prompt |
| 4 | Four sample complaints |
| 5 | Tool trace and structured handler comparison |
| Extensions | 2, 3, 4, and 5 only |

---
## Step 1: Setup

Add `OPENAI_API_KEY=...` to a `.env` file next to this notebook.

In [1]:
# Install required packages (uncomment to run)
!pip install -U langchain langchain-openai python-dotenv

**What:** Load the API key from `.env` and initialize `gpt-4o-mini` without displaying the key.


In [2]:
import os
import random
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

load_dotenv()

# Make sure OPENAI_API_KEY is set in your .env file
print("OpenAI API Key loaded:", "OPENAI_API_KEY" in os.environ)

model = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0.7)

OpenAI API Key loaded: True


**Done when:** the key prints `True` and there are no import errors.

---
## Step 2: Create Creative Tools

Every tool needs:
1. the `@tool` decorator
2. a clear docstring (the agent reads it to choose the tool)

### 2a. `consult_demogorgon`

**What:** Keep the instructor-provided Demogorgon tool and its three random replies.


In [3]:
@tool
def consult_demogorgon(complaint: str) -> str:
    """Consult the Demogorgon about a complaint involving the Upside Down.
    Returns a chaotic, unreliable opinion."""
    responses = [
        f"The Demogorgon sniffs '{complaint}' and roars. It seems... hungry, not helpful.",
        f"The Demogorgon tilts its petal-face at '{complaint}'. Interpreted as: 'turn off the lights'.",
        f"The Demogorgon ignores '{complaint}' and walks back into the wall. Classic.",
    ]
    return random.choice(responses)

### 2b. `check_hawkins_records`

- Dictionary with 4 keys: `portal`, `monsters`, `psychics`, `electricity`
- Return the record whose key appears in the query (case-insensitive)
- No match → `"No record found ..."`

**What:** Search the four fixed Hawkins record keys and return an explicit fallback.

In [4]:
# Match a known topic word in the query, ignoring case.
@tool
def check_hawkins_records(query: str) -> str:
    """Search the four topic-keyed Hawkins records for a key in the query."""
    records = {
        "portal": "Portal log: a gate was reported in Hawkins Lab's basement; the report recommends keeping distance.",
        "monsters": "Monster log: Demogorgon and Demodog sightings are listed; this record does not identify ordinary animals as monsters.",
        "psychics": "Psychics log: reports describe telekinetic activity; the log does not confirm a cause for any individual event.",
        "electricity": "Electricity log: flickering lights and voltage drops have been reported near unusual activity; cause not confirmed.",
    }
    query_lower = query.lower()
    for key, record in records.items():
        if key.lower() in query_lower:
            return record
    return f"No record found for query: {query}"


### 2c. `cast_interdimensional_spell` [COMPLETED]

- Args: `problem`, `creativity_level="medium"`
- Map `low/medium/high` → `1/2/3`
- 4 spell strings that include the problem
- `random.sample()` N spells, join with `"\n"`

**What:** Select one, two, or three problem-specific spells with `random.sample`.


In [5]:
# Randomly choose a number of problem-specific spells from four options.
@tool
def cast_interdimensional_spell(problem: str, creativity_level: str = "medium") -> str:
    """Suggest one to three playful spells for the stated problem."""
    multipliers = {"low": 1, "medium": 2, "high": 3}
    count = multipliers.get(creativity_level.lower(), 2)
    spells = [
        f"The Mirror Maze: redirect {problem} through a harmless mirror loop.",
        f"The Pocket Portal: shrink {problem} into a labeled lunchbox for one hour.",
        f"The Cosmic Confetti: surround {problem} with distracting, non-sticky stardust.",
        f"The Reverse Echo: send {problem} back as a polite note to its point of origin.",
    ]
    selected_spells = random.sample(spells, k=min(count, len(spells)))
    return "\n".join(selected_spells)


### 2d. `gather_party_wisdom` [COMPLETED]

- Dictionary with 4 keys, one per party member (Mike, Dustin, Lucas, Will)
- Return the match (case-insensitive)
- No match → generic "the party huddles..." answer

**What:** Return one of four party responses when a topic word appears, otherwise huddle.


In [6]:
# Return one party member's topic response, or a generic huddle if no key matches.
@tool
def gather_party_wisdom(question: str) -> str:
    """Find party advice by topic word, with a huddle fallback."""
    responses = {
        "plan": "Mike: Make a plan, agree on a signal, and stay together.",
        "science": "Dustin: Check the evidence first; a strange reading is still just a reading.",
        "danger": "Lucas: Keep a safe distance and have an exit ready.",
        "upside down": "Will: Trust your instincts, but compare them with what the records say.",
    }
    question_lower = question.lower()
    for topic, response in responses.items():
        if topic.lower() in question_lower:
            return response
    return "The party huddles around the map and agrees to compare notes before acting."


**What:** Call every tool directly, including one no-record query, to inspect its behavior.


In [7]:
# Invoke each tool directly so its input and return value are easy to inspect.
print(consult_demogorgon.invoke({"complaint": "My lights keep flickering"}))
print(check_hawkins_records.invoke({"query": "tell me about the portal"}))
print(cast_interdimensional_spell.invoke({"problem": "Demodogs in my garden", "creativity_level": "high"}))
print(gather_party_wisdom.invoke({"question": "what is the plan?"}))
print(check_hawkins_records.invoke({"query": "my neighbor's cat"}))


The Demogorgon sniffs 'My lights keep flickering' and roars. It seems... hungry, not helpful.
Portal log: a gate was reported in Hawkins Lab's basement; the report recommends keeping distance.
The Reverse Echo: send Demodogs in my garden back as a polite note to its point of origin.
The Mirror Maze: redirect Demodogs in my garden through a harmless mirror loop.
The Pocket Portal: shrink Demodogs in my garden into a labeled lunchbox for one hour.
Mike: Make a plan, agree on a signal, and stay together.
No record found for query: my neighbor's cat


**Done when:** all 4 tools return text, and a query with no matching key returns your fallback.

---
## Step 3: Create the Agent
System prompt must:
- encourage creative tool use
- force at least one tool call
- say: answer **only** from tool results
- ask the agent to list the tools it used

**Hint:** `create_agent(model=..., tools=..., system_prompt=...)` in `langchain_integration.ipynb`.

**What:** Register the four tools and create the modern LangChain agent with a grounding prompt.


In [8]:
# Registering tools.
tools = [
    consult_demogorgon,
    check_hawkins_records,
    cast_interdimensional_spell,
    gather_party_wisdom,
]

# Ask for creativity while restricting factual claims to returned tool text.
system_prompt = """You are the NormalObjects creative complaint handler.
Use the available tools creatively and in any order that helps. For every complaint,
make at least one tool call. Answer only from the results returned by tools. If a
record tool says 'No record found', say 'No record found' and do not invent a cause
or diagnosis. Treat the Demogorgon's opinion as an opinion, not verified evidence.
Write each factual or suggested claim as a short bullet and append the supporting
[tool_name]. Do not cite a tool unless its output supports that claim. End with a
' tools used:' line listing only tools actually called.""".replace("tools", "Tools")

agent = create_agent(model=model, tools=tools, system_prompt=system_prompt)
print([tool.name for tool in tools])


['consult_demogorgon', 'check_hawkins_records', 'cast_interdimensional_spell', 'gather_party_wisdom']


**Done when:** the agent is created and prints the 4 tool names.

---
## Step 4: Test with Sample Complaints

Use `config={"recursion_limit": 10}` to stop loops.

**What:** Invoke the agent on all four template complaints and save full traces and elapsed time.


In [9]:
complaints = [
    "My Christmas lights keep flickering and I hear noises from the wall. What is going on?",
    "There is a weird portal in my basement. What's the plan?",
    "A Demodog ate my homework. I need a creative solution, creativity level high!",
    "My neighbor's cat is acting weird. Is it from the Upside Down?",  # no record exists
]

# Run each sample complaint once and retain full messages for later tracing.
from time import perf_counter

results = []
agent_elapsed = []
for complaint in complaints:
    started = perf_counter()
    result = agent.invoke(
        {"messages": [HumanMessage(content=complaint)]},
        config={"recursion_limit": 10},
    )
    agent_elapsed.append(perf_counter() - started)
    results.append(result)
    print(f"\nComplaint: {complaint}")
    print(f"Answer: {result['messages'][-1].content}")



Complaint: My Christmas lights keep flickering and I hear noises from the wall. What is going on?
Answer: - There are no records found regarding Christmas lights flickering. [check_hawkins_records]
- There are no records found regarding noises from the wall. [check_hawkins_records]
- The Demogorgon suggests to "turn off the lights." [consult_demogorgon]

Tools used: check_hawkins_records, consult_demogorgon

Complaint: There is a weird portal in my basement. What's the plan?
Answer: - A gate was reported in Hawkins Lab's basement, and the report recommends keeping distance. [check_hawkins_records]
- The party suggests comparing notes before taking any action regarding the weird portal. [gather_party_wisdom]

Tools used: check_hawkins_records, gather_party_wisdom

Complaint: A Demodog ate my homework. I need a creative solution, creativity level high!
Answer: Here are some creative spells to address your situation with the Demodog that ate your homework:

- **The Reverse Echo**: Send "

---
## Step 5: Analyze Agent Behavior [COMPLETED]

### 5a. Track tool usage

For each result: print the tool order (`a → b → c`). Then count total uses per tool.

**Hint:** AI messages have `.tool_calls`; use `collections.Counter`.

**What:** Print each complaint's tool order and aggregate calls by tool name.


In [10]:
# Read tool calls from AI messages and count their names across all complaints.
from collections import Counter

tool_counts = Counter()
for index, result in enumerate(results, start=1):
    order = []
    for message in result["messages"]:
        if message.type == "ai":
            order.extend(call["name"] for call in message.tool_calls)
    tool_counts.update(order)
    print(f"Complaint {index} tool order: {' → '.join(order)}")
print("Total tool calls:", dict(tool_counts))


Complaint 1 tool order: check_hawkins_records → check_hawkins_records → consult_demogorgon
Complaint 2 tool order: check_hawkins_records → gather_party_wisdom
Complaint 3 tool order: cast_interdimensional_spell
Complaint 4 tool order: consult_demogorgon → check_hawkins_records
Total tool calls: {'check_hawkins_records': 4, 'consult_demogorgon': 2, 'gather_party_wisdom': 1, 'cast_interdimensional_spell': 1}


### 5b. Trace one answer (grounding)

For one result, print every tool call, its args, and what it returned.

**Hint:** messages with `msg.type == "tool"` hold the tool output.

**What:** Pair each AI tool call with its output by call ID, then label final-answer claims.


In [11]:
# Pair every requested tool call with its ToolMessage using tool_call_id.
trace_result = results[0]
trace_messages = trace_result["messages"]
outputs_by_id = {
    message.tool_call_id: message.content
    for message in trace_messages
    if message.type == "tool"
}

for message in trace_messages:
    if message.type == "ai":
        for call in message.tool_calls:
            print(f"Tool: {call['name']}")
            print(f"Call ID: {call['id']}")
            print(f"Args: {call['args']}")
            print(f"Output for this ID: {outputs_by_id.get(call['id'], 'No matching ToolMessage')}")

final_answer = trace_messages[-1].content
print("\nFinal answer:")
print(final_answer)
print("\nClaim attribution (a tool citation is a claim; compare it with the paired output above):")
for line in final_answer.splitlines():
    claim = line.strip(" -*\t")
    if not claim or claim.lower().startswith("tools used:"):
        continue
    cited_tools = [tool.name for tool in tools if f"[{tool.name}]" in claim]
    if cited_tools:
        print(f"FROM TOOL {', '.join(cited_tools)}: {claim}")
    else:
        print(f"NOT ATTRIBUTED TO A TOOL (treat as unsupported): {claim}")


Tool: check_hawkins_records
Call ID: call_cY1gcJye6dFT0jMrXU1ulMZd
Args: {'query': 'Christmas lights flickering'}
Output for this ID: No record found for query: Christmas lights flickering
Tool: check_hawkins_records
Call ID: call_7OfY0OTvFugpHZSIKUX83HuH
Args: {'query': 'noises from wall'}
Output for this ID: No record found for query: noises from wall
Tool: consult_demogorgon
Call ID: call_WvxXSPoTZQv6keYz6jRdxS9K
Args: {'complaint': 'My Christmas lights keep flickering and I hear noises from the wall.'}
Output for this ID: The Demogorgon tilts its petal-face at 'My Christmas lights keep flickering and I hear noises from the wall.'. Interpreted as: 'turn off the lights'.

Final answer:
- There are no records found regarding Christmas lights flickering. [check_hawkins_records]
- There are no records found regarding noises from the wall. [check_hawkins_records]
- The Demogorgon suggests to "turn off the lights." [consult_demogorgon]

Tools used: check_hawkins_records, consult_demogorgo

**Check:** mark each claim in the final answer as either from a tool or invented.

### 5c. Compare with a structured approach

Write `structured_handler(complaint)`: always calls records → party → spell (`low`), no LLM.

**What:** Run records, party, and a low spell in a fixed order without an LLM.


In [12]:
# This fixed handler calls records, party, then one low-creativity spell; it uses no LLM.
def structured_handler(complaint: str) -> str:
    record = check_hawkins_records.invoke({"query": complaint})
    party = gather_party_wisdom.invoke({"question": complaint})
    spell = cast_interdimensional_spell.invoke(
        {"problem": complaint, "creativity_level": "low"}
    )
    return f"Records: {record}\nParty: {party}\nSpell: {spell}"

print(structured_handler(complaints[1]))


Records: Portal log: a gate was reported in Hawkins Lab's basement; the report recommends keeping distance.
Party: Mike: Make a plan, agree on a signal, and stay together.
Spell: The Reverse Echo: send There is a weird portal in my basement. What's the plan? back as a polite note to its point of origin.


| | Agent | Structured |
|---|---|---|
| Same tools every run? | No; the model chooses tools for each complaint. | Yes; records, party, and spell run every time. |
| Tool order predictable? | No; order can vary. | Yes; records → party → spell. |
| Can invent facts? | It can still produce unsupported wording despite the prompt; inspect the trace. | No LLM is used, though tool text itself is limited to its hardcoded data. |
| Easy to trace? | Yes, if AI calls are matched to ToolMessages by call ID. | Yes; the fixed calls and returned values are explicit. |
| Best for which client case? | Open-ended complaints where varied suggestions are useful and reviewed. | Cases needing a predictable, repeatable sequence. |

---
## Extensions 2–5

These extensions follow the brief: an LLM-backed version of the existing Demogorgon tool, in-memory thread memory, a local Gradio evidence view, and a small performance comparison. Extension 1 and Optional Stretch are not included.

## Extensions 2

**What:** Redefine `consult_demogorgon` under the same tool name so it calls the model internally.

**Why:** Extension 2 asks for dynamic tool responses while preserving the required four-tool set.

In [13]:
# Keep the same tool name and use one model call inside its implementation.
@tool
def consult_demogorgon(complaint: str) -> str:
    """Ask the Demogorgon for a short, chaotic opinion about a complaint."""
    response = model.invoke(
        "Give one short, chaotic, clearly unreliable NormalObjects-style opinion "
        f"about this complaint: {complaint}"
    )
    return response.content

print("Extension tool name:", consult_demogorgon.name)
extension_tools = [
    consult_demogorgon,
    check_hawkins_records,
    cast_interdimensional_spell,
    gather_party_wisdom,
]
extension_agent = create_agent(model=model, tools=extension_tools, system_prompt=system_prompt)
print(consult_demogorgon.invoke({"complaint": "My bicycle bell rings by itself."}))


Extension tool name: consult_demogorgon
Clearly, your bicycle bell is not malfunctioning; it’s actually a sentient being trying to communicate with you! It’s probably trying to warn you about an impending invasion of rogue squirrels planning to steal your snacks. Embrace the chaos—your bike is now a messenger of the animal kingdom!


## Extensions 3

**What:** Attach `InMemorySaver` and use two thread IDs to keep prior complaints separate.

**Why:** Extension 3 requires conversation memory and two independent threads.


In [14]:
# This small memory demo isolates thread behavior from complaint-tool behavior.
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver(),
    system_prompt="Use conversation history to answer questions about the earlier message.",
)

thread_red = {"configurable": {"thread_id": "complaint-red"}, "recursion_limit": 10}
thread_blue = {"configurable": {"thread_id": "complaint-blue"}, "recursion_limit": 10}

memory_agent.invoke(
    {"messages": [HumanMessage(content="Remember my complaint: Christmas lights flicker.")]},
    config=thread_red,
)
red_followup = memory_agent.invoke(
    {"messages": [HumanMessage(content="What complaint did I mention?")]},
    config=thread_red,
)
memory_agent.invoke(
    {"messages": [HumanMessage(content="Remember my complaint: a portal is in the basement.")]},
    config=thread_blue,
)
blue_followup = memory_agent.invoke(
    {"messages": [HumanMessage(content="What complaint did I mention?")]},
    config=thread_blue,
)
print("Red thread:", red_followup["messages"][-1].content)
print("Blue thread:", blue_followup["messages"][-1].content)


Red thread: You mentioned a complaint about your Christmas lights flickering.
Blue thread: You mentioned that a portal is in the basement, which seems to be a concern for you. Would you like to discuss it further?


## Extensions 4

**What:** Serve the complaint agent's answer beside a table of matched tool calls and outputs.

**Why:** Extension 4 asks for a local-only Gradio interface that exposes evidence with the answer.


In [15]:
# Return the final answer and call-ID-matched evidence rows to Gradio.
import gradio as gr


def answer_with_evidence(complaint: str):
    result = extension_agent.invoke(
        {"messages": [HumanMessage(content=complaint)]},
        config={"recursion_limit": 10},
    )
    messages = result["messages"]
    outputs = {
        message.tool_call_id: message.content
        for message in messages
        if message.type == "tool"
    }
    rows = []
    for message in messages:
        if message.type == "ai":
            for call in message.tool_calls:
                rows.append([
                    call["name"],
                    str(call["args"]),
                    outputs.get(call["id"], "No matching ToolMessage"),
                ])
    return messages[-1].content, rows


demo = gr.Interface(
    fn=answer_with_evidence,
    inputs=gr.Textbox(label="Complaint"),
    outputs=[
        gr.Textbox(label="Answer"),
        gr.Dataframe(headers=["Tool", "Args", "Output"], label="Tool evidence"),
    ],
    title="NormalObjects Complaint Handler",
)
demo.launch(server_name="127.0.0.1", share=False, prevent_thread_lock=True)


/opt/anaconda3/envs/bootcamp-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Extensions 5

**What:** Summarize elapsed time, LLM calls, tool calls, repeats, and tokens for agent and fixed paths.

**Why:** Extension 5 requests per-complaint measurements and an agent-versus-structured comparison.


In [16]:
# Read token metadata from AI messages; report N/A if the provider omitted it.
def agent_metrics(result):
    llm_calls = 0
    tool_names = []
    token_totals = {"input": 0, "output": 0, "total": 0}
    saw_usage = False
    for message in result["messages"]:
        if message.type == "ai":
            llm_calls += 1
            tool_names.extend(call["name"] for call in message.tool_calls)
            usage = getattr(message, "usage_metadata", None) or {}
            if usage:
                saw_usage = True
                token_totals["input"] += usage.get("input_tokens", 0)
                token_totals["output"] += usage.get("output_tokens", 0)
                token_totals["total"] += usage.get("total_tokens", 0)
    counts = Counter(tool_names)
    repeats = sum(max(0, count - 1) for count in counts.values())
    tokens = token_totals if saw_usage else "N/A (provider returned no usage metadata)"
    return llm_calls, len(tool_names), repeats, tokens


print("Agent versus structured handler (one row per complaint)")
for index, complaint in enumerate(complaints):
    llm_calls, tool_calls, repeats, tokens = agent_metrics(results[index])
    structured_started = perf_counter()
    structured_handler(complaint)
    structured_seconds = perf_counter() - structured_started
    print(f"\nComplaint {index + 1}: {complaint}")
    print(
        f"Agent: time={agent_elapsed[index]:.2f}s, LLM calls={llm_calls}, "
        f"tool calls={tool_calls}, repeated calls={repeats}, tokens={tokens}"
    )
    print(
        f"Structured: time={structured_seconds:.4f}s, LLM calls=0, "
        "tool calls=3, repeated calls=0, tokens=0 (no LLM)"
    )


Agent versus structured handler (one row per complaint)

Complaint 1: My Christmas lights keep flickering and I hear noises from the wall. What is going on?
Agent: time=5.35s, LLM calls=3, tool calls=3, repeated calls=1, tokens={'input': 1189, 'output': 158, 'total': 1347}
Structured: time=0.0012s, LLM calls=0, tool calls=3, repeated calls=0, tokens=0 (no LLM)

Complaint 2: There is a weird portal in my basement. What's the plan?
Agent: time=3.07s, LLM calls=2, tool calls=2, repeated calls=0, tokens={'input': 717, 'output': 122, 'total': 839}
Structured: time=0.0006s, LLM calls=0, tool calls=3, repeated calls=0, tokens=0 (no LLM)

Complaint 3: A Demodog ate my homework. I need a creative solution, creativity level high!
Agent: time=3.69s, LLM calls=2, tool calls=1, repeated calls=0, tokens={'input': 724, 'output': 136, 'total': 860}
Structured: time=0.0004s, LLM calls=0, tool calls=3, repeated calls=0, tokens=0 (no LLM)

Complaint 4: My neighbor's cat is acting weird. Is it from the Up